# Loading an XML Corpus

This template loads just the normalized tokens of each XML document in a folder, without token metadata, into segments based on `<s>` (i.e. sentence-like) elements. To load a verse corpus, load `<l>` (and <`lg>`) instead, as in the [_n_-grams demo notebook](https://github.com/langeslag/ehtc/blob/main/demo/n-grams.ipynb); to include metadata, adapt the template to represent each document as a list of dictionaries (cf. [parsed_corpora.ipynb](https://github.com/langeslag/ehtc/blob/main/demo/parsed_corpora.ipynb)).

In [1]:
from pathlib import Path
from lxml import etree
from git import Repo
from collections import Counter

We'll ascertain the ECHOE repository has been cloned so we have XML documents to work with:

In [2]:
# HTTPS clone point:
remote = 'https://github.com/ECHOEProject/echoe.git'
# Desired target folder name:
local = Path.cwd().parent / 'corpora' / 'echoe'
# Only clone if the target folder doesn't already exist:
if not(local.exists()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

In [ ]:
# Normalization matrix (Tironian notes are language-dependent and are thus handled separately below):
substitutions = {
    'ę': 'æ',
    'ƿ': 'w',
    'ẏ': 'y',
    'ſ': 's',
    '': 's', # Using the glyph for descending s, instead of the unicode key point
    'v': 'u',
    'j': 'i',
    'ꝛ': 'r',
    '&': 'et',
    '\uf149': 'þ',
    '\ue337': 'þ',
    '·': '',
    ' ': '',
    '\n': '',
    '\u2028': ''
}

# Token normalization:
def normalize(token):
    # Lowercase:
    token = token.lower()
    for k,v in substitutions.items():
        # Carry out replacements:
        token = token.replace(k, v)
    return token

# Discarding unwanted elements:
def simplify(branch):
    discard = ['abbr', 'am', 'sic', 'del', 'note', 'surplus', 'orig', 'fw']  
    # Now we define their text nodes as empty strings:
    query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
    for hit in branch.iter(query):
        for element in hit.iter():
            element.text = ''
            element.tail = ''
    return branch

In [4]:
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)
corpus_folder = local / 'xml'
corpus = dict()
for file in corpus_folder.glob('*.xml'):
    basename = file.name[:-4]
    tree = etree.parse(file, parser=parser)
    root = simplify(tree.getroot())
    segments = dict()
    for segment in root.iter('{http://www.tei-c.org/ns/1.0}s'):
        # Remember to switch to the default XML namespace to access @xml:id or @xml:lang attributes!
        identifier = segment.get('{http://www.w3.org/XML/1998/namespace}id')
        tokens = []
        for token in segment.iter('{http://www.tei-c.org/ns/1.0}w'):
            if token.get('{http://www.w3.org/XML/1998/namespace}lang') == 'la' or token.xpath('ancestor::*[@xml:lang][1]/@xml:lang')[0] == 'la':
                token_string = normalize(etree.tostring(token, method='text', encoding='unicode')).replace('⁊', 'et').replace('⹒', 'et')
            else:
                token_string = normalize(etree.tostring(token, method='text', encoding='unicode')).replace('⁊', 'and').replace('⹒', 'and')
            # If a word element is marked as the last part of a word, add its text content to the preceding token:
            if token.get('part') == 'F':
                position = len(tokens)-1
                tokens[position] = tokens[position] + token_string
            else:
                tokens.append(token_string)
        segments[identifier] = tokens
    corpus[basename] = segments
        

In [5]:
corpus['018.40']['s18.40.66']

['conuertimini',
 'ad',
 'me',
 'in',
 'toto',
 'corde',
 'uestro',
 'in',
 'ieiunio',
 'et',
 'fletu',
 'et',
 'planc',
 'peccatorum',
 'uestrorum',
 'et',
 'ego',
 'reuertar',
 'ad',
 'uos']

To ensure normalization is complete, we'll run a character counter:

In [6]:
all_tokens = list()
for doc in corpus.values():
    for sentence in doc.values():
        for token in sentence:
            all_tokens.append(token)
corpus_string = ''.join(all_tokens)
counter = Counter(corpus_string)
counter.most_common()

[('e', 331082),
 ('n', 230652),
 ('a', 211001),
 ('o', 140348),
 ('s', 131035),
 ('d', 128304),
 ('i', 122820),
 ('r', 114038),
 ('t', 94452),
 ('l', 92747),
 ('h', 88181),
 ('þ', 82963),
 ('m', 82574),
 ('g', 79404),
 ('æ', 73253),
 ('w', 67296),
 ('u', 65501),
 ('c', 61345),
 ('f', 55266),
 ('ð', 47671),
 ('y', 36437),
 ('b', 28738),
 ('p', 9111),
 ('x', 1425),
 ('q', 639),
 ('k', 211),
 ('z', 163),
 ('ī', 8)]